# Baseline Methods and Ablation Studies

**Paper:** *Constrained Multi-Objective Particle Swarm Optimization for Sensitivity-Aware Pruning of Lung Cancer Histopathology Networks* 

## 0. Setup

In [ ]:
import copy, json, math, os, random, time
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.utils.prune as prune
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, models, transforms
from tqdm.auto import tqdm

print(f'PyTorch: {torch.__version__}')
print(f'CUDA   : {torch.cuda.is_available()}')

## 1. Configuration


In [ ]:
# ---------------- Must match 01_mopso_pipeline_optionB.ipynb ----------------
SEED = 42
DATASET = 'LungHist700'
DATA_DIR = {
    'LungHist700': 'data/LungHist700',
    'LC25000':     'data/LC25000',
}[DATASET]
NUM_CLASSES = 3

TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15

IMG_SIZE   = 224
BATCH_SIZE = 32
NUM_WORKERS = 0

EPOCHS_PRETRAIN = 15         
EPOCHS_FINETUNE = 10
LR_PRETRAIN    = 1e-4
LR_FINETUNE    = 1e-5
WEIGHT_DECAY   = 1e-4

PROBING_RATES   = [0.3, 0.5, 0.7]
DROP_TOLERANCE  = 0.05
NCAL_CAP        = 500

N_PARTICLES, N_ITERATIONS, ARCHIVE_SIZE = 30, 50, 50  
INERTIA_W, COGNITIVE_C1, SOCIAL_C2 = 0.5, 1.5, 1.5
PENALTY_LAMBDA, SENS_THRESHOLD_TAU = 10.0, 0.90
NOMINAL_SMAX = 0.95

DETERMINISTIC_CUDNN = True

RESULTS_DIR = Path(f'results/{DATASET}_seed{SEED}_scalar')
CKPT_DIR    = Path(f'checkpoints/{DATASET}_seed{SEED}_scalar')
RESULTS_FILE = RESULTS_DIR / 'all_results.json'
assert RESULTS_FILE.exists(), (
    f'{RESULTS_FILE} not found. Run 01_mopso_pipeline_optionB.ipynb first.'
)

In [ ]:
def set_global_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    if DETERMINISTIC_CUDNN:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True

set_global_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

with open(RESULTS_FILE) as f:
    results = json.load(f)
print(f'Loaded results with keys: {list(results.keys())}')

def save_results():
    with open(RESULTS_FILE, 'w') as f:
        json.dump(results, f, indent=2, default=str)

def log_result(exp_id, data):
    results[exp_id] = data
    save_results()
    print(f'[{exp_id}] saved to {RESULTS_FILE}')

### 0.2 Data and backbone


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE), transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

LUNGHIST_SUPERCLASS = {
    'aca_bd': 'aca', 'aca_md': 'aca', 'aca_pd': 'aca',
    'scc_bd': 'scc', 'scc_md': 'scc', 'scc_pd': 'scc',
    'nor':    'nor',
}

class ImageFolderWithMapping(datasets.ImageFolder):
    def __init__(self, root, transform=None, class_mapping=None):
        super().__init__(root, transform=transform)
        if class_mapping is None:
            self.super_classes = self.classes; return
        super_class_names = sorted(set(class_mapping.values()))
        name_to_idx = {n: i for i, n in enumerate(super_class_names)}
        self.super_classes = super_class_names
        self.samples = [(p, name_to_idx[class_mapping[self.classes[old]]])
                        for p, old in self.samples]
        self.targets = [s[1] for s in self.samples]

def build_image_folder(root, transform):
    if DATASET == 'LungHist700':
        return ImageFolderWithMapping(root, transform=transform,
                                       class_mapping=LUNGHIST_SUPERCLASS)
    return datasets.ImageFolder(root, transform=transform)

full_train = build_image_folder(DATA_DIR, transform=train_tf)
full_eval  = build_image_folder(DATA_DIR, transform=eval_tf)
CLASS_NAMES = (full_train.super_classes
               if hasattr(full_train, 'super_classes') else full_train.classes)

n_total = len(full_train)
n_train = int(TRAIN_RATIO * n_total); n_val = int(VAL_RATIO * n_total)
n_test = n_total - n_train - n_val
gen = torch.Generator().manual_seed(SEED)
train_idx, val_idx, test_idx = random_split(
    range(n_total), [n_train, n_val, n_test], generator=gen)
train_idx = list(train_idx); val_idx = list(val_idx); test_idx = list(test_idx)

train_set = Subset(full_train, train_idx)
val_set   = Subset(full_eval,  val_idx)
test_set  = Subset(full_eval,  test_idx)

train_targets = np.array([full_train.targets[i] for i in train_idx])
class_counts = np.bincount(train_targets, minlength=NUM_CLASSES)
CLASS_WEIGHTS = torch.tensor(
    class_counts.sum() / (NUM_CLASSES * class_counts), dtype=torch.float32
).to(DEVICE)

def make_loader(ds, shuffle=False):
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True)

train_loader = make_loader(train_set, shuffle=True)
val_loader   = make_loader(val_set)
test_loader  = make_loader(test_set)

NCAL = min(NCAL_CAP, len(val_set))
cal_indices = np.random.RandomState(SEED).choice(len(val_set), size=NCAL,
                                                  replace=False).tolist()
cal_subset = Subset(val_set, cal_indices)
cal_loader = make_loader(cal_subset)
print(f'Train/Val/Test: {len(train_set)}/{len(val_set)}/{len(test_set)}')
print(f'Calibration: {NCAL}')

### 1.2 Results container

In [ ]:
RESULTS_FILE = RESULTS_DIR / 'all_results.json'
results = {}

def save_results():
    with open(RESULTS_FILE, 'w') as f:
        json.dump(results, f, indent=2, default=str)

def log_result(exp_id, data):
    results[exp_id] = data
    save_results()
    print(f'[{exp_id}] saved to {RESULTS_FILE}')

## 2. Data loading and splits

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

assert os.path.isdir(DATA_DIR), f'Dataset folder not found: {DATA_DIR}'

LUNGHIST_SUPERCLASS = {
    'aca_bd': 'aca', 'aca_md': 'aca', 'aca_pd': 'aca',
    'scc_bd': 'scc', 'scc_md': 'scc', 'scc_pd': 'scc',
    'nor':    'nor',
}

class ImageFolderWithMapping(datasets.ImageFolder):
    def __init__(self, root, transform=None, class_mapping=None):
        super().__init__(root, transform=transform)
        if class_mapping is None:
            self.super_classes = self.classes
            return
        super_class_names = sorted(set(class_mapping.values()))
        name_to_idx = {n: i for i, n in enumerate(super_class_names)}
        self.super_classes = super_class_names
        self.samples = [
            (p, name_to_idx[class_mapping[self.classes[old]]])
            for p, old in self.samples
        ]
        self.targets = [s[1] for s in self.samples]

def build_image_folder(root, transform):
    if DATASET == 'LungHist700':
        return ImageFolderWithMapping(root, transform=transform,
                                       class_mapping=LUNGHIST_SUPERCLASS)
    return datasets.ImageFolder(root, transform=transform)

full_train = build_image_folder(DATA_DIR, transform=train_tf)
full_eval  = build_image_folder(DATA_DIR, transform=eval_tf)
CLASS_NAMES = (full_train.super_classes
               if hasattr(full_train, 'super_classes')
               else full_train.classes)
print(f'Dataset size: {len(full_train)}')
print(f'Classes     : {CLASS_NAMES}')

In [ ]:
def build_resnet50(num_classes=NUM_CLASSES, pretrained=True):
    weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
    m = models.resnet50(weights=weights)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m.to(DEVICE)

def get_prunable_layers(model):
    out = []
    for n, m in model.named_modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            if isinstance(m, nn.Linear) and m.out_features == NUM_CLASSES:
                continue
            out.append((n, m))
    return out

def apply_sparsity_vector(model, sparsity_vec):
    layers = get_prunable_layers(model)
    assert len(layers) == len(sparsity_vec)
    for (n, m), s in zip(layers, sparsity_vec):
        s = float(max(0.0, min(NOMINAL_SMAX, s)))
        if s > 0:
            prune.l1_unstructured(m, name='weight', amount=s)

def fuse_pruning_masks(model):
    for n, m in get_prunable_layers(model):
        try: prune.remove(m, 'weight')
        except ValueError: pass

def actual_sparsity(model):
    tot, z = 0, 0
    for _, m in get_prunable_layers(model):
        tot += m.weight.numel(); z += (m.weight == 0).sum().item()
    return z / max(1, tot)

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    loss_sum, corr, tot = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()
        out = model(x); loss = criterion(out, y); loss.backward(); optimizer.step()
        loss_sum += loss.item() * x.size(0)
        corr += (out.argmax(1) == y).sum().item(); tot += x.size(0)
    return loss_sum / tot, corr / tot

def compute_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    pcr = cm.diagonal() / cm.sum(axis=1).clip(min=1)
    out = {'accuracy': float(accuracy_score(y_true, y_pred)),
           'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
           'sens_min': float(pcr.min())}
    for i, c in enumerate(CLASS_NAMES):
        out[f'sens_{c}'] = float(pcr[i])
    return out

@torch.no_grad()
def eval_model(model, loader):
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        p = model(x).argmax(1).cpu().tolist()
        ys.extend(y.tolist()); ps.extend(p)
    return compute_metrics(np.array(ys), np.array(ps))

def get_layer_info(model):
    params, flops_list = [], []
    handles, out_shapes = [], {}
    def hook(name):
        def h(mod, inp, out): out_shapes[name] = tuple(out.shape)
        return h
    for n, m in get_prunable_layers(model):
        handles.append(m.register_forward_hook(hook(n)))
    model.eval()
    with torch.no_grad():
        _ = model(torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE))
    for h in handles: h.remove()
    for n, m in get_prunable_layers(model):
        params.append(m.weight.numel())
        s = out_shapes[n]
        if isinstance(m, nn.Conv2d):
            c_in = m.weight.shape[1]
            k_h, k_w = m.weight.shape[2], m.weight.shape[3]
            c_out = m.out_channels
            _, _, H, W = s
            fl = 2 * (c_in // m.groups) * k_h * k_w * c_out * H * W
        else:
            fl = 2 * m.in_features * m.out_features
        flops_list.append(int(fl))
    return params, flops_list

def flops_params_ratio(sparsity_vec, lp, lf):
    s = np.asarray(sparsity_vec); p = np.asarray(lp); f = np.asarray(lf)
    return float((f * (1 - s)).sum() / f.sum()), float((p * (1 - s)).sum() / p.sum())

In [ ]:
dense_path = CKPT_DIR / 'dense_resnet50.pt'
assert dense_path.exists(), f'{dense_path} not found. Run notebook 01 first.'

model = build_resnet50(pretrained=False)
model.load_state_dict(torch.load(dense_path, map_location=DEVICE))
LAYER_PARAMS, LAYER_FLOPS = get_layer_info(model)
N_PRUNABLE = len(LAYER_PARAMS)
criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)

SMAX_PER_LAYER = results['E2']['s_max_per_layer']
assert len(SMAX_PER_LAYER) == N_PRUNABLE
print(f'Dense model loaded. N_PRUNABLE = {N_PRUNABLE}.')

### 0.3 Shared prune-and-fine-tune helper (same as Option A)

In [ ]:
def prune_and_finetune(sparsity_vec, epochs=EPOCHS_FINETUNE):
    pruned = build_resnet50(pretrained=False)
    pruned.load_state_dict(torch.load(dense_path, map_location=DEVICE))
    apply_sparsity_vector(pruned, sparsity_vec)
    opt = torch.optim.AdamW(pruned.parameters(),
                             lr=LR_FINETUNE, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best_val, best_state = 0.0, None
    for e in range(epochs):
        train_one_epoch(pruned, train_loader, opt, criterion)
        vm = eval_model(pruned, val_loader); sched.step()
        if vm['accuracy'] > best_val:
            best_val = vm['accuracy']
            best_state = copy.deepcopy(pruned.state_dict())
    pruned.load_state_dict(best_state)
    fuse_pruning_masks(pruned)
    tm = eval_model(pruned, test_loader)
    fr, pr = flops_params_ratio(sparsity_vec, LAYER_PARAMS, LAYER_FLOPS)
    as_ = actual_sparsity(pruned)
    out = {'test_metrics': tm, 'flops_ratio': fr, 'params_ratio': pr,
           'actual_sparsity': as_}
    del pruned
    return out

## 1. Baselines


### 1.1 Global magnitude

In [ ]:
print('Baseline: Magnitude')
mag_results = {}
for s in [0.3, 0.5, 0.7]:
    print(f'--- s = {s} ---')
    out = prune_and_finetune([s] * N_PRUNABLE)
    mag_results[f's={s}'] = out
    print(f'  Acc {out["test_metrics"]["accuracy"]:.4f} | '
          f'sens_min {out["test_metrics"]["sens_min"]:.4f}')
log_result('E3_magnitude', mag_results)

### 1.2 Global Magnitude Pruning (GMP)

In [ ]:
print('Baseline: GMP')
def gmp_sparsity_vector(model, target_global_sparsity):
    all_abs, layer_sizes = [], []
    for _, m in get_prunable_layers(model):
        w = m.weight.detach().abs().cpu().flatten()
        all_abs.append(w.numpy()); layer_sizes.append(w.numel())
    thresh = np.quantile(np.concatenate(all_abs), target_global_sparsity)
    return [float((arr <= thresh).mean()) for arr in all_abs]

gmp_results = {}
for s in [0.5, 0.7]:
    print(f'--- GMP target s = {s} ---')
    sv = gmp_sparsity_vector(model, s)
    out = prune_and_finetune(sv)
    gmp_results[f's={s}'] = out
log_result('E4_gmp', gmp_results)

### 1.3 SNIP

In [ ]:
print('Baseline: SNIP')

def snip_sparsity_vector(model, target_s, n_batches=1):
    model = copy.deepcopy(model); model.train()
    for _, m in get_prunable_layers(model):
        m.weight.requires_grad_(True)
    scores = {name: torch.zeros_like(m.weight) for name, m in get_prunable_layers(model)}
    criterion_local = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)
    it = iter(train_loader)
    for _ in range(n_batches):
        try: x, y = next(it)
        except StopIteration: break
        x, y = x.to(DEVICE), y.to(DEVICE)
        model.zero_grad()
        out = model(x); loss = criterion_local(out, y); loss.backward()
        for name, m in get_prunable_layers(model):
            if m.weight.grad is not None:
                scores[name] += (m.weight * m.weight.grad).abs().detach()
    masks = {}
    for name, m in get_prunable_layers(model):
        s_arr = scores[name].flatten()
        k = int(target_s * s_arr.numel())
        if k == 0:
            masks[name] = torch.ones_like(m.weight)
        else:
            threshold = torch.topk(s_arr, k, largest=False).values.max()
            masks[name] = (scores[name] > threshold).float()
    del model
    return masks

def prune_snip_and_finetune(target_s, epochs=EPOCHS_FINETUNE):
    base = build_resnet50(pretrained=False)
    base.load_state_dict(torch.load(dense_path, map_location=DEVICE))
    masks = snip_sparsity_vector(base, target_s, n_batches=1)
    for name, m in get_prunable_layers(base):
        prune.custom_from_mask(m, name='weight', mask=masks[name])
    opt = torch.optim.AdamW(base.parameters(), lr=LR_FINETUNE,
                             weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best_val, best_state = 0.0, None
    for e in range(epochs):
        train_one_epoch(base, train_loader, opt, criterion)
        vm = eval_model(base, val_loader); sched.step()
        if vm['accuracy'] > best_val:
            best_val = vm['accuracy']
            best_state = copy.deepcopy(base.state_dict())
    base.load_state_dict(best_state)
    fuse_pruning_masks(base)
    sv = [target_s] * N_PRUNABLE
    fr, pr = flops_params_ratio(sv, LAYER_PARAMS, LAYER_FLOPS)
    out = {'test_metrics': eval_model(base, test_loader),
           'flops_ratio': fr, 'params_ratio': pr,
           'actual_sparsity': actual_sparsity(base)}
    del base
    return out

snip_results = {}
for s in [0.3, 0.5, 0.7]:
    print(f'--- SNIP s = {s} ---')
    snip_results[f's={s}'] = prune_snip_and_finetune(s)
log_result('E5_snip', snip_results)

## 2. Ablation studies

In [ ]:
def proxy_fitness(sparsity_vec):
    trial = copy.deepcopy(model)
    apply_sparsity_vector(trial, sparsity_vec)
    m = eval_model(trial, cal_loader)
    fuse_pruning_masks(trial)
    fr, pr = flops_params_ratio(sparsity_vec, LAYER_PARAMS, LAYER_FLOPS)
    del trial
    return {'accuracy': m['accuracy'], 'sens_min': m['sens_min'],
            'flops_ratio': fr, 'params_ratio': pr}


def dominates(a, b):
    a, b = np.asarray(a), np.asarray(b)
    return bool(np.all(a <= b) and np.any(a < b))


class ScalarSumAblatableMOPSO:
    """Ablation-aware scalar-sum MOPSO.

    - Multi-objective mode: pbest and archive truncation on sum(F).
    - Single-objective mode: pbest and archive on F[0] directly.
    - Constraint on/off: controls whether the penalty is added to F[0].
    """

    def __init__(self, n_particles, n_dims, smax_vec,
                 use_constraint=True, multi_objective=True,
                 w=INERTIA_W, c1=COGNITIVE_C1, c2=SOCIAL_C2,
                 archive_size=ARCHIVE_SIZE,
                 penalty_lambda=PENALTY_LAMBDA,
                 sens_threshold=SENS_THRESHOLD_TAU, seed=SEED):
        self.n_particles, self.n_dims = n_particles, n_dims
        self.smax = np.array(smax_vec)
        self.w, self.c1, self.c2 = w, c1, c2
        self.archive_size = archive_size
        self.lam, self.tau = penalty_lambda, sens_threshold
        self.use_constraint = use_constraint
        self.multi_objective = multi_objective
        self.rng_init = np.random.RandomState(seed)
        self.rng_loop = np.random.RandomState(seed + 1)
        self.X = self.rng_init.uniform(0, 1, (n_particles, n_dims)) * self.smax[None, :]
        self.V = self.rng_init.uniform(-0.1, 0.1, (n_particles, n_dims))
        self.pbest_X = self.X.copy()
        self.pbest_F = [None] * n_particles
        self.archive = []

    def evaluate(self, x):
        x = np.clip(x, 0.0, self.smax)
        p = proxy_fitness(x.tolist())
        penalty = self.lam * max(0.0, self.tau - p['sens_min']) ** 2 if self.use_constraint else 0.0
        if self.multi_objective:
            F = (1.0 - p['accuracy'] + penalty, p['flops_ratio'], p['params_ratio'])
        else:
            F = (1.0 - p['accuracy'] + penalty,)
        return F, p, x

    def update_pbest(self, i, F_new, x_new):
        """Scalar-sum update."""
        if self.pbest_F[i] is None or sum(F_new) < sum(self.pbest_F[i]):
            self.pbest_X[i] = x_new.copy(); self.pbest_F[i] = F_new

    def update_archive(self, x_new, F_new, p_new):
        if not self.multi_objective:
            if not self.archive or F_new[0] < self.archive[0][1][0]:
                self.archive = [(x_new.copy(), F_new, p_new)]
            return
        for _, aF, _ in self.archive:
            if dominates(aF, F_new): return
        self.archive = [(ax, aF, ap) for ax, aF, ap in self.archive
                        if not dominates(F_new, aF)]
        self.archive.append((x_new.copy(), F_new, p_new))
        if len(self.archive) > self.archive_size:
            # Scalar-sum truncation
            self.archive = sorted(self.archive, key=lambda t: sum(t[1]))[:self.archive_size]

    def pick_leader(self):
        if not self.archive:
            return self.X[self.rng_loop.randint(self.n_particles)]
        return self.archive[self.rng_loop.randint(len(self.archive))][0]

    def run(self, n_iter):
        for it in tqdm(range(n_iter), desc='Ablation MOPSO (scalar)'):
            for i in range(self.n_particles):
                F, p, x_clip = self.evaluate(self.X[i])
                self.X[i] = x_clip
                self.update_pbest(i, F, x_clip)
                self.update_archive(x_clip, F, p)
            for i in range(self.n_particles):
                r1 = self.rng_loop.uniform(0, 1, self.n_dims)
                r2 = self.rng_loop.uniform(0, 1, self.n_dims)
                leader = self.pick_leader()
                self.V[i] = (self.w * self.V[i]
                              + self.c1 * r1 * (self.pbest_X[i] - self.X[i])
                              + self.c2 * r2 * (leader - self.X[i]))
                self.V[i] = np.clip(self.V[i], -0.2, 0.2)
                self.X[i] = np.clip(self.X[i] + self.V[i], 0.0, self.smax)
        return self.archive

### 2.1 Ablation 1: medical sensitivity constraint

In [ ]:
print('Ablation 1: MOPSO without medical constraint (scalar-sum)')
t0 = time.time()
abl1 = ScalarSumAblatableMOPSO(
    n_particles=N_PARTICLES, n_dims=N_PRUNABLE,
    smax_vec=SMAX_PER_LAYER,
    use_constraint=False, multi_objective=True,
)
arch1 = abl1.run(N_ITERATIONS)
F_mat = np.array([t[1] for t in arch1])
idx_hc = int(np.argmin((F_mat[:, 1] + F_mat[:, 2]) / 2))
x_hc, F_hc, p_hc = arch1[idx_hc]
out_abl1 = prune_and_finetune(x_hc.tolist())
print(f'  Acc {out_abl1["test_metrics"]["accuracy"]:.4f} | '
      f'sens_min {out_abl1["test_metrics"]["sens_min"]:.4f} | '
      f'FLOPs {out_abl1["flops_ratio"]:.3f}')
log_result('E_ablation1', {
    'no_constraint': out_abl1, 'archive_size': len(arch1),
    'wall_time_s': time.time() - t0,
})

### 2.2 Ablation 2: multi-objective vs single-objective

In [ ]:
print('Ablation 2: Single-objective PSO (scalar-sum)')
t0 = time.time()
abl2 = ScalarSumAblatableMOPSO(
    n_particles=N_PARTICLES, n_dims=N_PRUNABLE,
    smax_vec=SMAX_PER_LAYER,
    use_constraint=True, multi_objective=False,
)
arch2 = abl2.run(N_ITERATIONS)
x_so, F_so, p_so = arch2[0]
out_abl2 = prune_and_finetune(x_so.tolist())
print(f'  Acc {out_abl2["test_metrics"]["accuracy"]:.4f} | '
      f'FLOPs {out_abl2["flops_ratio"]:.3f} | Params {out_abl2["params_ratio"]:.3f}')
log_result('E_ablation2', {
    'single_objective': out_abl2, 'wall_time_s': time.time() - t0,
})

### 2.3 Ablation 3: proxy vs full evaluation

In [ ]:
print('Ablation 3: Proxy vs full-evaluation MOPSO (scalar-sum)')
FULL_EVAL_PARTICLES = 6
FULL_EVAL_ITERS     = 3

def full_eval_fitness(sparsity_vec):
    pruned = build_resnet50(pretrained=False)
    pruned.load_state_dict(torch.load(dense_path, map_location=DEVICE))
    apply_sparsity_vector(pruned, sparsity_vec)
    opt = torch.optim.AdamW(pruned.parameters(), lr=LR_FINETUNE)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_FINETUNE)
    for e in range(EPOCHS_FINETUNE):
        train_one_epoch(pruned, train_loader, opt, criterion); sched.step()
    cm = eval_model(pruned, cal_loader)
    fr, pr = flops_params_ratio(sparsity_vec, LAYER_PARAMS, LAYER_FLOPS)
    fuse_pruning_masks(pruned); del pruned
    return {'accuracy': cm['accuracy'], 'sens_min': cm['sens_min'],
            'flops_ratio': fr, 'params_ratio': pr}

print(f'Proxy MOPSO: N={N_PARTICLES}, T={N_ITERATIONS}')
t_proxy = time.time()
proxy_mopso = ScalarSumAblatableMOPSO(
    n_particles=N_PARTICLES, n_dims=N_PRUNABLE,
    smax_vec=SMAX_PER_LAYER,
    use_constraint=True, multi_objective=True,
)
proxy_arch = proxy_mopso.run(N_ITERATIONS)
proxy_wall = time.time() - t_proxy

_proxy_backup = proxy_fitness
globals()['proxy_fitness'] = full_eval_fitness

print(f'Full-eval MOPSO: N={FULL_EVAL_PARTICLES}, T={FULL_EVAL_ITERS}')
t_full = time.time()
full_mopso = ScalarSumAblatableMOPSO(
    n_particles=FULL_EVAL_PARTICLES, n_dims=N_PRUNABLE,
    smax_vec=SMAX_PER_LAYER,
    use_constraint=True, multi_objective=True,
)
full_arch = full_mopso.run(FULL_EVAL_ITERS)
full_wall = time.time() - t_full

globals()['proxy_fitness'] = _proxy_backup

speedup = full_wall / proxy_wall if proxy_wall > 0 else float('nan')
print(f'Speedup: {speedup:.2f}x')

log_result('E_ablation3', {
    'proxy_wall_s': proxy_wall, 'full_wall_s': full_wall, 'speedup': speedup,
    'proxy_archive': len(proxy_arch), 'full_archive': len(full_arch),
    'proxy_particles': N_PARTICLES, 'proxy_iters': N_ITERATIONS,
    'full_particles': FULL_EVAL_PARTICLES, 'full_iters': FULL_EVAL_ITERS,
})

## 3. Summary

In [ ]:
print('=' * 60)
print(f'Baselines and ablations complete for {DATASET}, seed {SEED} (scalar variant)')
print('=' * 60)
print(f'Results in: {RESULTS_FILE}')
for k in sorted(results.keys()):
    print(f'  - {k}')